Imports and configuration

In [2]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
import matplotlib.pyplot as plt



2025-12-25 21:32:46.181665: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766698366.376394      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766698366.431636      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766698366.882012      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766698366.882063      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766698366.882067      55 computation_placer.cc:177] computation placer alr

In [3]:
import os

DATA_DIR = "../input/flowers-recognition/flowers"
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42


In [4]:
import os
print(os.listdir("../input/flowers-recognition/flowers"))


['dandelion', 'daisy', 'sunflower', 'tulip', 'rose']


**Load dataset and create initial train + temp split** :
Here we create a 70% train / 30% temp split using image_dataset_from_directory.

In [5]:
# TensorFlow can only create TWO splits at a time.
# Here we ask for:
#   - validation_split = 0.3  → 30% held out
#   - subset="training"       → return the remaining 70%

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    labels="inferred",            # Use folder names as labels
    label_mode="int", #  (sparse labels)
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    shuffle=True,
    seed=SEED,
    validation_split=0.3,         # 30% reserved for temp
    subset="training"             # <-- This gives the 70% split
)


# This is the SAME function call, but:
#   subset="validation" → returns the 30% portion
# This TEMP dataset will later be split into:
#   - 15% validation
#   - 15% test

temp_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    labels="inferred",
    label_mode="int", # (sparse labels)
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    shuffle=True,
    seed=SEED,
    validation_split=0.3,         # Same 30% split
    subset="validation"           # <-- This gives the 30% split
)


class_names = train_ds.class_names
num_classes = len(class_names)
print("Classes:", class_names)


Found 4317 files belonging to 5 classes.
Using 3022 files for training.


I0000 00:00:1766698384.979101      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1766698384.983172      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Found 4317 files belonging to 5 classes.
Using 1295 files for validation.
Classes: ['daisy', 'dandelion', 'rose', 'sunflower', 'tulip']


**Split TEMP → VALIDATION (15%) + TEST (15%)**

In [6]:
# Count how many batches are in the 30% TEMP dataset
temp_size = tf.data.experimental.cardinality(temp_ds).numpy()

# Half of TEMP → 15% of total dataset
val_size = temp_size // 2

# First half of TEMP → validation set
val_ds = temp_ds.take(val_size)

# Second half of TEMP → test set
test_ds = temp_ds.skip(val_size)


**Prefetch for performance**

In [7]:
# Prefetching loads data in the background while the model trains
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)


**Verify the splits**

In [8]:
print("Train batches:", tf.data.experimental.cardinality(train_ds).numpy())
print("Validation batches:", tf.data.experimental.cardinality(val_ds).numpy())
print("Test batches:", tf.data.experimental.cardinality(test_ds).numpy())


Train batches: 95
Validation batches: 20
Test batches: 21


**Stage 2: Transfer Learning (TL) with MobileNetV2**

**Import MobileNetV2 (without the top classifier)**

In [9]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

# Load base model (pretrained on ImageNet)
base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,      # Remove ImageNet classifier
    weights="imagenet"
)

base_model.trainable = False   # Freeze for Transfer Learning


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


**Add your custom classifier head**

In [10]:
inputs = layers.Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(5, activation="softmax")(x)   # 5 flower classes

model = models.Model(inputs, outputs)


**Compile the model**

In [11]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy", # <-- sparse loss
    metrics=["accuracy"]
)


**Train Transfer Learning stage**

In [12]:
history_tl = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)


Epoch 1/5


I0000 00:00:1766698394.191906     128 service.cc:152] XLA service 0x7f08080042e0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1766698394.191945     128 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1766698394.191949     128 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1766698395.269068     128 cuda_dnn.cc:529] Loaded cuDNN version 91002
2025-12-25 21:33:23.472009: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-12-25 21:33:23.608986: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
I0000 00:00:1766698405.745370     128 device_co

94/95 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - accuracy: 0.2752 - loss: 1.8763

2025-12-25 21:33:39.863859: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-12-25 21:33:40.000772: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


95/95 ━━━━━━━━━━━━━━━━━━━━ 39s 230ms/step - accuracy: 0.2763 - loss: 1.8717 - val_accuracy: 0.4547 - val_loss: 1.3643
Epoch 2/5
95/95 ━━━━━━━━━━━━━━━━━━━━ 4s 38ms/step - accuracy: 0.4520 - loss: 1.3255 - val_accuracy: 0.5250 - val_loss: 1.2330
Epoch 3/5
95/95 ━━━━━━━━━━━━━━━━━━━━ 3s 36ms/step - accuracy: 0.5293 - loss: 1.2075 - val_accuracy: 0.5688 - val_loss: 1.1712
Epoch 4/5
95/95 ━━━━━━━━━━━━━━━━━━━━ 3s 36ms/step - accuracy: 0.5654 - loss: 1.1313 - val_accuracy: 0.5484 - val_loss: 1.2088
Epoch 5/5
95/95 ━━━━━━━━━━━━━━━━━━━━ 3s 36ms/step - accuracy: 0.5799 - loss: 1.0880 - val_accuracy: 0.5938 - val_loss: 1.1225


**Evaluate on the Test Set and Train**

In [13]:
# -----------------------------
# EVALUATE ON TRAIN SET
# -----------------------------
# This shows how well the model fits the data it learned from.
train_loss, train_acc = model.evaluate(train_ds)
print("Train Loss:", train_loss)
print("Train Accuracy:", train_acc)

# -----------------------------
# EVALUATE ON TEST SET
# -----------------------------
# This shows how well the model generalizes to unseen data.
test_loss, test_acc = model.evaluate(test_ds)
print("Test Loss:", test_loss)
print("Test Accuracy:", test_acc)

# -----------------------------
# INTERPRETATION GUIDE
# -----------------------------
# If Train Accuracy >> Test Accuracy → Overfitting
# If Train Accuracy ≈ Test Accuracy → Good generalization
# If Train Loss << Test Loss → Overfitting
# If Losses are similar → Healthy model


95/95 ━━━━━━━━━━━━━━━━━━━━ 6s 61ms/step - accuracy: 0.6554 - loss: 0.9570
Train Loss: 0.9485988020896912
Train Accuracy: 0.6581733822822571
20/21 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 0.6137 - loss: 1.0538

2025-12-25 21:36:57.223996: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-12-25 21:36:57.359814: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


21/21 ━━━━━━━━━━━━━━━━━━━━ 11s 530ms/step - accuracy: 0.6133 - loss: 1.0547
Test Loss: 1.0627949237823486
Test Accuracy: 0.609160304069519


**Fine‑Tuning Stage (unfreeze deeper layers)**

**Check numbers of layers of based model**

In [14]:
# Count how many layers are in the base model
print("Number of layers in base model:", len(base_model.layers))

# Optional: list them with indices
for i, layer in enumerate(base_model.layers):
    print(i, layer.name, layer.trainable)


Number of layers in base model: 154
0 input_layer False
1 Conv1 False
2 bn_Conv1 False
3 Conv1_relu False
4 expanded_conv_depthwise False
5 expanded_conv_depthwise_BN False
6 expanded_conv_depthwise_relu False
7 expanded_conv_project False
8 expanded_conv_project_BN False
9 block_1_expand False
10 block_1_expand_BN False
11 block_1_expand_relu False
12 block_1_pad False
13 block_1_depthwise False
14 block_1_depthwise_BN False
15 block_1_depthwise_relu False
16 block_1_project False
17 block_1_project_BN False
18 block_2_expand False
19 block_2_expand_BN False
20 block_2_expand_relu False
21 block_2_depthwise False
22 block_2_depthwise_BN False
23 block_2_depthwise_relu False
24 block_2_project False
25 block_2_project_BN False
26 block_2_add False
27 block_3_expand False
28 block_3_expand_BN False
29 block_3_expand_relu False
30 block_3_pad False
31 block_3_depthwise False
32 block_3_depthwise_BN False
33 block_3_depthwise_relu False
34 block_3_project False
35 block_3_project_BN False

**Unfreeze deeper layers for fine‑tuning**

In [15]:
# ---------------------------------------------------------
# FINE‑TUNING SETUP
# ---------------------------------------------------------
# We unfreeze the base MobileNetV2 model so deeper layers
# can be updated during training.
# BUT we keep the first ~100 layers frozen to avoid
# destroying low‑level ImageNet features (edges, textures).
# ---------------------------------------------------------

base_model.trainable = True   # Allow training on deeper layers

for layer in base_model.layers[:100]:
    layer.trainable = False   # Keep early layers frozen


**Re‑compile with a very small learning rate**

In [16]:
# ---------------------------------------------------------
# RE‑COMPILE MODEL FOR FINE‑TUNING
# ---------------------------------------------------------
# Fine‑tuning must use a VERY small learning rate.
# Otherwise, the pretrained weights get damaged.
# ---------------------------------------------------------

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),  # tiny LR for stability
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


**Train the fine‑tuning stage**

In [17]:
# ---------------------------------------------------------
# FINE‑TUNING TRAINING
# ---------------------------------------------------------
# Now the model updates deeper convolutional layers.
# This usually improves accuracy by 2–10%.
# ---------------------------------------------------------

history_ft = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)


Epoch 1/5
95/95 ━━━━━━━━━━━━━━━━━━━━ 40s 202ms/step - accuracy: 0.3628 - loss: 1.9375 - val_accuracy: 0.3938 - val_loss: 1.6545
Epoch 2/5
95/95 ━━━━━━━━━━━━━━━━━━━━ 5s 47ms/step - accuracy: 0.5112 - loss: 1.2674 - val_accuracy: 0.3609 - val_loss: 1.8424
Epoch 3/5
95/95 ━━━━━━━━━━━━━━━━━━━━ 4s 47ms/step - accuracy: 0.5807 - loss: 1.0873 - val_accuracy: 0.3562 - val_loss: 1.9661
Epoch 4/5
95/95 ━━━━━━━━━━━━━━━━━━━━ 5s 48ms/step - accuracy: 0.6467 - loss: 0.9446 - val_accuracy: 0.3656 - val_loss: 1.9969
Epoch 5/5
95/95 ━━━━━━━━━━━━━━━━━━━━ 5s 49ms/step - accuracy: 0.6780 - loss: 0.8604 - val_accuracy: 0.3938 - val_loss: 1.8128


**Evaluate on TRAIN + TEST to check overfitting**

In [18]:
# ---------------------------------------------------------
# EVALUATE ON TRAIN SET
# ---------------------------------------------------------
# Shows how well the model fits the data it learned from.
# ---------------------------------------------------------
train_loss, train_acc = model.evaluate(train_ds)
print("Train Loss:", train_loss)
print("Train Accuracy:", train_acc)

# ---------------------------------------------------------
# EVALUATE ON TEST SET
# ---------------------------------------------------------
# Shows how well the model generalizes to unseen data.
# ---------------------------------------------------------
test_loss, test_acc = model.evaluate(test_ds)
print("Test Loss:", test_loss)
print("Test Accuracy:", test_acc)

# ---------------------------------------------------------
# INTERPRETATION
# ---------------------------------------------------------
# If Train Accuracy >> Test Accuracy → Overfitting
# If Train Accuracy ≈ Test Accuracy → Good generalization
# ---------------------------------------------------------


95/95 ━━━━━━━━━━━━━━━━━━━━ 6s 63ms/step - accuracy: 0.4553 - loss: 1.5653
Train Loss: 1.547768235206604
Train Accuracy: 0.4649238884449005
21/21 ━━━━━━━━━━━━━━━━━━━━ 4s 188ms/step - accuracy: 0.4082 - loss: 1.7011
Test Loss: 1.650848388671875
Test Accuracy: 0.42137405276298523
